In [ ]:
from dotenv import load_dotenv
from pydantic import BaseModel, EmailStr
from pydantic_settings import BaseSettings, SettingsConfigDict

load_dotenv()


class ConfigSettings(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=".env.prod", env_file_encoding="utf-8", extra="ignore"
    )
    model: str
    openai_api_key: str
    x_api_key: str
    environment: str
    hello: str


# User Schema
class User(BaseModel):
    first_name: str
    last_name: str
    email: EmailStr

In [ ]:
settings = ConfigSettings().model_dump()

# model = settings.get("model")
# x_api_key = settings.get("x_api_key")
# openai_api_key = settings.get("openai_api_key")
# rprint(settings)

In [ ]:
from crewai import LLM, Agent, Crew, Process, Task
from crewai.knowledge.source.string_knowledge_source import StringKnowledgeSource

# Create a knowledge source
content = "Users name is John. He is 30 years old and lives in San Francisco."
string_source = StringKnowledgeSource(content=content)

# Create an LLM with a temperature of 0 to ensure deterministic outputs
llm = LLM(model="gpt-4o-mini", temperature=0)

# Create an agent with the knowledge store
agent = Agent(
    role="About User",
    goal="You know everything about the user.",
    backstory="You are a master at understanding people and their preferences.",
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

task = Task(
    description="Answer the following questions about the user: {question}",
    expected_output="An answer to the question.",
    agent=agent,
)

crew = Crew(
    agents=[agent],
    tasks=[task],
    verbose=True,
    process=Process.sequential,
    knowledge_sources=[string_source],  # Enable knowledge by adding the sources here
)

result = crew.kickoff(
    inputs={"question": "What city does John live in and how old is he?"}
)

In [ ]:
from crewai.rag.config.utils import clear_rag_config, get_rag_client, set_rag_config

# Qdrant
from crewai.rag.qdrant.config import QdrantConfig

set_rag_config(QdrantConfig())
qdrant_client = get_rag_client()

# Example operations (same API for any provider)
client = qdrant_client  # or chromadb_client
client.create_collection(collection_name="docs")
client.add_documents(
    collection_name="docs",
    documents=[{"id": "1", "content": "CrewAI enables collaborative AI agents."}],
)
results = client.search(collection_name="docs", query="collaborative agents", limit=3)

clear_rag_config()  # optional reset

In [ ]:
import uuid
from datetime import datetime, timezone
from enum import Enum
from typing import Optional

from pydantic import BaseModel, EmailStr
from sqlmodel import Field, SQLModel

# ---------------------------------------------------------------------------
# Enums
# ---------------------------------------------------------------------------


class FileType(str, Enum):
    """Allowed transcript file types."""

    TXT = "text/plain"
    PDF = "application/pdf"
    DOCX = "application/vnd.openxmlformats-officedocument.wordprocessingml.document"
    VTT = "text/vtt"
    SRT = "application/x-subrip"


# ---------------------------------------------------------------------------
# Database Models (SQLModel — maps to DB table)
# ---------------------------------------------------------------------------


class FileUploads(SQLModel, table=True):
    """Stores metadata for uploaded transcription files."""

    id: uuid.UUID = Field(
        default_factory=uuid.uuid4,
        primary_key=True,
        index=True,
    )
    file_name: str = Field(
        ...,
        max_length=255,
        description="Original name of the uploaded file",
    )
    file_type: str = Field(
        ...,
        description="MIME type of the uploaded file e.g. text/plain",
    )
    file_size: int = Field(
        ...,
        description="File size in bytes",
    )
    file_path: str = Field(
        ...,
        description="Relative path or S3 key where the file is stored",
    )
    uploaded_at: datetime = Field(
        default_factory=lambda: datetime.now(timezone.utc),
        description="UTC timestamp of when the file was uploaded",
    )


# ---------------------------------------------------------------------------
# Request Models (Pydantic — used in FastAPI route bodies)
# ---------------------------------------------------------------------------


class AgentRequest(BaseModel):
    """Request body for triggering the timesheet agent crew."""

    employee_fullnames: list[str] = Field(
        ...,
        description="List of full names to search for in the transcription",
        examples=[["Jane Doe", "John Smith"]],
    )
    email_address: Optional[EmailStr] = Field(
        default=None,
        description="Optional email address to send the generated timesheet to",
    )
    file_upload_id: uuid.UUID = Field(
        ...,
        description="UUID of the previously uploaded FileUploads record to process",
    )

    class Config:
        json_schema_extra = {
            "example": {
                "employee_fullnames": ["Jane Doe", "John Smith"],
                "email_address": "jane.doe@company.com",
                "file_upload_id": "3fa85f64-5717-4562-b3fc-2c963f66afa6",
            }
        }


# ---------------------------------------------------------------------------
# Response Models (Pydantic — returned by FastAPI routes)
# ---------------------------------------------------------------------------


class FileUploadResponse(BaseModel):
    """Returned after a successful file upload."""

    id: uuid.UUID
    file_name: str
    file_type: str
    file_size: int
    file_path: str
    uploaded_at: datetime

    class Config:
        from_attributes = True


class AgentTaskStatus(str, Enum):
    """Possible states for an agent run."""

    PENDING = "pending"
    RUNNING = "running"
    COMPLETED = "completed"
    FAILED = "failed"


class AgentResponse(BaseModel):
    """Returned after triggering the timesheet agent crew."""

    request_id: uuid.UUID = Field(
        default_factory=uuid.uuid4,
        description="Unique ID for this agent run — use to poll for status",
    )
    status: AgentTaskStatus = Field(
        default=AgentTaskStatus.PENDING,
        description="Current status of the agent crew run",
    )
    employee_fullnames: list[str]
    file_upload_id: uuid.UUID
    message: str = Field(
        default="Agent crew has been queued and will begin processing shortly.",
    )
    output_file: Optional[str] = Field(
        default=None,
        description="Path or URL to the generated timesheet CSV once completed",
    )